# BTW 2025: Validate generated vote entries

This notebook checks the finished 2025 JSON files independently. It validates the shared record structure, the 2025 demographic domains, duplicate rows, non-negative values, and conservation of the official constituency, party, vote-type, and postal/in-person totals.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "scripts").is_dir() and (candidate / "package.json").is_file():
            return candidate
    raise RuntimeError("Run this notebook inside the repository.")

ROOT = find_repository_root()
sys.path.insert(0, str(ROOT))
DISTRICT_RESULTS_CSV = ROOT / "scripts/data/btw25_wbz_ergebnisse.csv"
GENERATED_DIRECTORY = ROOT / "scripts/data/generated/btw2025"
FIRST_VOTES_JSON = GENERATED_DIRECTORY / "first_votes.json"
SECOND_VOTES_JSON = GENERATED_DIRECTORY / "second_votes.json"

from scripts.election_data.btw2025 import (
    BTW2025_AGE_GROUPS,
    read_polling_district_csv,
    reshape_polling_district_votes,
)
from scripts.election_data.notebook_steps import (
    aggregate_to_constituencies,
    inspect_district_rows,
    normalize_district_rows,
    select_usable_district_rows,
)


In [ ]:
def load_vote_entries(path: Path) -> pd.DataFrame:
    if not path.is_file():
        raise FileNotFoundError(f"Generated JSON does not exist: {path}")
    with path.open("r", encoding="utf-8") as handle:
        return pd.DataFrame.from_records(json.load(handle))

first_entries = load_vote_entries(FIRST_VOTES_JSON)
second_entries = load_vote_entries(SECOND_VOTES_JSON)
entries = pd.concat([first_entries, second_entries], ignore_index=True)

expected_columns = {
    "districtId", "state", "gender", "ageGroup",
    "party", "voteType", "electionMethod", "votes",
}
key_columns = [
    "districtId", "state", "gender", "ageGroup",
    "party", "voteType", "electionMethod",
]
assert set(first_entries.columns) == expected_columns
assert set(second_entries.columns) == expected_columns
assert set(first_entries["voteType"]) == {"1"}
assert set(second_entries["voteType"]) == {"2"}
assert set(entries["gender"]).issubset({"m", "w"})
assert set(entries["ageGroup"]) == set(BTW2025_AGE_GROUPS)
assert set(entries["electionMethod"]).issubset({"in-person", "postal"})
assert entries["votes"].notna().all()
assert (entries["votes"] >= 0).all()
assert not entries.duplicated(key_columns).any()

print(f"First-vote entries: {len(first_entries):,}")
print(f"Second-vote entries: {len(second_entries):,}")
print("Age groups:", sorted(entries["ageGroup"].unique()))
print('gender="m" represents the published combined group "m|d|o".')


In [ ]:
raw_districts = read_polling_district_csv(DISTRICT_RESULTS_CSV)
diagnostics = inspect_district_rows(raw_districts)
usable_districts = select_usable_district_rows(raw_districts, diagnostics)
normalized_districts = normalize_district_rows(usable_districts)

official_totals = pd.concat(
    [
        aggregate_to_constituencies(
            reshape_polling_district_votes(normalized_districts, vote_type="1")
        ),
        aggregate_to_constituencies(
            reshape_polling_district_votes(normalized_districts, vote_type="2")
        ),
    ],
    ignore_index=True,
)

source_keys = ["districtId", "state", "party", "voteType", "electionMethod"]
generated_totals = entries.groupby(source_keys, as_index=False)["votes"].sum()
comparison = official_totals.merge(
    generated_totals,
    on=source_keys,
    how="outer",
    suffixes=("Official", "Generated"),
).fillna(0.0)
comparison["absoluteError"] = (
    comparison["votesOfficial"] - comparison["votesGenerated"]
).abs()
display(comparison.sort_values("absoluteError", ascending=False).head(20))
print("Maximum absolute error:", comparison["absoluteError"].max())
assert comparison["absoluteError"].max() <= 1e-6

nationwide = (
    entries.groupby(["voteType", "party"], as_index=False)["votes"]
    .sum()
    .sort_values(["voteType", "votes"], ascending=[True, False])
)
nationwide["share"] = nationwide["votes"] / nationwide.groupby("voteType")[
    "votes"
].transform("sum")
display(nationwide.groupby("voteType").head(20))

demographic = entries.groupby(
    ["voteType", "gender", "ageGroup"], as_index=False
)["votes"].sum()
display(demographic.sort_values(["voteType", "gender", "ageGroup"]))


When all assertions pass, every exact official source total is preserved. The detailed demographic rows remain a state-profile interpolation rather than observed individual ballots.
